In [10]:
import pandas as pd
import requests
import io
from dataexp import Dataexp
from google.cloud import storage
from datetime import datetime

In [60]:
URL = "https://data.wa.gov/resource/f6w7-q2d2.csv"

LIMIT = 100_000
OFFSET = 0

chunks = []
dtype = {'vin_1_10': 'string',
         'county': 'string',
         'city': 'string',
         'state': 'string',
         'zip_code': 'string',
         'model_year': 'int32',
         'make': 'string',
         'model': 'string',
         'ev_type': 'string',
         'cafv_type': 'string',
         'electric_range': 'float32',
         'legislative_district': 'string',
         'dol_vehicle_id': 'string', 
         'geocoded_column': 'string', 
         'electric_utility': 'string',
         '_2020_census_tract': 'string'}

with requests.Session() as session:
    while True:
        response = session.get(
            URL,
            params={
                "$limit": LIMIT,
                "$offset": OFFSET,
            },
            timeout=600,
        )
        response.raise_for_status()

        chunk = pd.read_csv(io.BytesIO(response.content), dtype=dtype)

        if chunk.empty:
            print("No more rows to download.")
            break

        chunks.append(chunk)

        OFFSET += len(chunk)

        print(f"Retrieved {OFFSET:,} rows")

        if len(chunk) < LIMIT:
            print("No more rows to download.")
            break

df = pd.concat(chunks, ignore_index=True)
parquet_bytes = df.to_parquet()

Retrieved 100,000 rows
Retrieved 200,000 rows
Retrieved 299,705 rows
No more rows to download.


In [61]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 299705 entries, 0 to 299704
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   vin_1_10              299705 non-null  string 
 1   county                299694 non-null  string 
 2   city                  299694 non-null  string 
 3   state                 299705 non-null  string 
 4   zip_code              299694 non-null  string 
 5   model_year            299705 non-null  int32  
 6   make                  299705 non-null  string 
 7   model                 299705 non-null  string 
 8   ev_type               299705 non-null  string 
 9   cafv_type             299705 non-null  string 
 10  electric_range        299679 non-null  float32
 11  legislative_district  298915 non-null  string 
 12  dol_vehicle_id        299705 non-null  string 
 13  geocoded_column       299685 non-null  string 
 14  electric_utility      299694 non-null  string 
 15  _2020_censu

In [28]:
now = datetime.now()

storage_client = storage.Client(project='data-engineer-project-506916')
bucket = storage_client.bucket('ev_population_data_pipeline_bucket')

blob = bucket.blob(f'{now.year}-{now.strftime('%b')} ev_population_data.parquet')
blob.upload_from_string(parquet_bytes)
print('Successfully upload data to Google cloud storage')

Successfully upload data to Google cloud storage


In [29]:
blob = bucket.blob(f'{now.year}-{now.strftime('%b')} ev_population_data.parquet')
contents = io.BytesIO(blob.download_as_bytes())

df = pd.read_parquet(contents)
print('Finished Download and Load to DataFrame')

Finished Download and Load to DataFrame


In [62]:
df_copy = df.copy()

In [106]:
df_copy.head()

,vin_1_10,county,city,state,zip_code,model_year,make,model,ev_type,cafv_type,electric_range,legislative_district,geocoded_column,electric_utility,_2020_census_tract
dol_vehicle_id,,,,,,,,,,,,,,,
168526641,1n4az1cp3k,king,shoreline,wa,98155,2019,nissan,leaf,battery electric vehicle (bev),clean alternative fuel vehicle eligible,150.0,32,point (-122.30304 47.75494),city of seattle - (wa)|city of tacoma - (wa),53033020500
9418181,wby8p4c51k,king,kirkland,wa,98033,2019,bmw,i3,plug-in hybrid electric vehicle (phev),clean alternative fuel vehicle eligible,126.0,48,point (-122.2066 47.67887),puget sound energy inc||city of tacoma - (wa),53033022703
172853495,3fa6p0su8g,snohomish,lynnwood,wa,98036,2016,ford,fusion,plug-in hybrid electric vehicle (phev),not eligible due to low battery range,19.0,1,point (-122.29245 47.82557),puget sound energy inc,53061051932
204768175,5yj3e1eb3n,yakima,wapato,wa,98951,2022,tesla,model 3,battery electric vehicle (bev),eligibility unknown as battery range has not b...,0.0,15,point (-120.42083 46.44779),pacificorp,53077940008
269262920,wby1z2c53f,king,renton,wa,98055,2015,bmw,i3,battery electric vehicle (bev),clean alternative fuel vehicle eligible,81.0,11,point (-122.19488 47.44034),puget sound energy inc||city of tacoma - (wa),53033025805


In [72]:
exp = Dataexp(df_copy)

In [73]:
exp.dtypes

{'O': [('vin_1_10', 'string'),
  ('county', 'string'),
  ('city', 'string'),
  ('state', 'string'),
  ('zip_code', 'string'),
  ('make', 'string'),
  ('model', 'string'),
  ('ev_type',
   'category',
   ['battery electric vehicle (bev)',
    'plug-in hybrid electric vehicle (phev)']),
  ('cafv_type',
   'category',
   ['clean alternative fuel vehicle eligible',
    'eligibility unknown as battery range has not been researched',
    'not eligible due to low battery range']),
  ('legislative_district', 'string'),
  ('geocoded_column', 'string'),
  ('electric_utility', 'string'),
  ('_2020_census_tract', 'string')],
 'i': [('model_year', 'int32')],
 'f': [('electric_range', 'float32')]}

In [35]:
categories = {'cafv_type': 
                  {'clean alternative fuel vehicle eligible',
                   'eligibility unknown as battery range has not been researched',
                   'not eligible due to low battery range', 
                   'not defined'},
              'ev_type': 
                  {'battery electric vehicle (bev)', 
                   'plug-in hybrid electric vehicle (phev)',
                   'not defined'}}

In [69]:
df_copy = df.apply(lambda x: x.str.lower().str.strip() if x.dtype.name == 'string' else x)
df_copy.set_index('dol_vehicle_id', inplace=True)

for key, val in categories.items():
    diff = set(df_copy[key].unique()) - val
    print(diff)
    
    df_copy.loc[df_copy[key].isin(diff), key] = 'not defined'
    print('Success')

    df_copy[key] = df_copy[key].astype('category')

null = df.isnull().any(axis=1)

null_df = df[null]
clean_df = df[~null]

set()
Success
set()
Success


In [74]:
exp.null

vin_1_10                  0
county                   11
city                     11
state                     0
zip_code                 11
model_year                0
make                      0
model                     0
ev_type                   0
cafv_type                 0
electric_range           26
legislative_district    790
geocoded_column          20
electric_utility         11
_2020_census_tract       11
dtype: int64

In [75]:
df[df.isnull().any(axis=1)].sample(50)

,vin_1_10,county,city,state,zip_code,model_year,make,model,ev_type,cafv_type,electric_range,legislative_district,dol_vehicle_id,geocoded_column,electric_utility,_2020_census_tract
295590,5YJXCDE26L,Harris,Houston,TX,77002,2020,TESLA,MODEL X,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,289.0,<NA>,2606453,POINT (-95.36952 29.76078),NON WASHINGTON STATE ELECTRIC UTILITY,48201980700
297115,7SAYGDEF5N,Montgomery,Montgomery,AL,36113,2022,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,<NA>,289782230,POINT (-86.34036 32.3804),NON WASHINGTON STATE ELECTRIC UTILITY,01101000900
279972,5YJ3E1EA2P,San Diego,San Diego,CA,92109,2023,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,<NA>,235160782,POINT (-117.25114 32.79099),NON WASHINGTON STATE ELECTRIC UTILITY,06073007602
299681,7SAYGDEF5N,Broward,Fort Lauderdale,FL,33312,2022,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,<NA>,209149209,POINT (-80.2002 26.087),NON WASHINGTON STATE ELECTRIC UTILITY,12011080406
295895,5YJ3E1EB4M,Santa Clara,San Jose,CA,95126,2021,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,<NA>,268060964,POINT (-121.91016 37.32799),NON WASHINGTON STATE ELECTRIC UTILITY,06085502203
283710,5YJ3E1EA7R,Wayne,Pikeville,NC,27863,2024,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,<NA>,272809477,POINT (-77.98164 35.49815),NON WASHINGTON STATE ELECTRIC UTILITY,37191000303
285357,5YJSA1E50N,Cook,Chicago,IL,60659,2022,TESLA,MODEL S,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,<NA>,195204551,POINT (-87.69926 41.99044),NON WASHINGTON STATE ELECTRIC UTILITY,17031020802
249693,7SAYGDEE4N,El Paso,Monument,CO,80132,2022,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,<NA>,195647805,POINT (-104.87157 39.09153),NON WASHINGTON STATE ELECTRIC UTILITY,08041007301
190011,7SAYGDEE5R,Clark,Las Vegas,NV,89148,2024,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,<NA>,265090031,POINT (-115.30437 36.07015),NON WASHINGTON STATE ELECTRIC UTILITY,32003005860
297829,JTDKAMFP4M,Clark,Las Vegas,NV,89130,2021,TOYOTA,PRIUS PRIME (PHEV),Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25.0,<NA>,259285869,POINT (-115.23121 36.24776),NON WASHINGTON STATE ELECTRIC UTILITY,32003003428


In [8]:
df.duplicated(subset=['vin_1_10']).sum()

np.int64(276125)

In [9]:
((df.isnull().any(axis=1).sum() / len(df)) * 100.0).round(2)

np.float64(0.27)

In [10]:
null_df = df[df.isnull().any(axis=1)]

In [11]:
clean_df = df[~df.isnull().any(axis=1)]

In [12]:
df_state = set(df['state'].unique())

In [15]:
df[df['state'].isin(df_state - states)]

,vin_1_10,county,city,state,zip_code,model_year,make,model,ev_type,cafv_type,electric_range,legislative_district,dol_vehicle_id,geocoded_column,electric_utility,_2020_census_tract
372,1G1FX6S08J,NaN,NaN,BC,NaN,2018,CHEVROLET,BOLT EV,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,238.0,NaN,272897614,NaN,NaN,NaN
243135,5YJ3E1EA6K,NaN,NaN,BC,NaN,2019,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,220.0,NaN,323559204,NaN,NaN,NaN
252984,3FA6P0SU7E,NaN,NaN,AE,NaN,2014,FORD,FUSION,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,19.0,NaN,255045136,NaN,NaN,NaN
257860,WVGTMPE23M,NaN,NaN,AE,NaN,2021,VOLKSWAGEN,ID.4,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,NaN,179339058,NaN,NaN,NaN
265661,5YJ3E1EA5S,NaN,NaN,BC,NaN,2025,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,NaN,280877355,NaN,NaN,NaN
269448,7SAYGDEF8N,NaN,NaN,QC,NaN,2022,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,NaN,211959226,NaN,NaN,NaN
272169,7SAYGDEE9N,NaN,NaN,BC,NaN,2022,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,NaN,224286394,NaN,NaN,NaN
276895,5YJXCAE24H,NaN,NaN,BC,NaN,2017,TESLA,MODEL X,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,200.0,NaN,159850029,NaN,NaN,NaN
293578,JTDKAMFP9N,NaN,NaN,AE,NaN,2022,TOYOTA,PRIUS PRIME (PHEV),Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25.0,NaN,198537741,NaN,NaN,NaN


In [11]:
df.duplicated(['dol_vehicle_id']).sum()

np.int64(0)

In [16]:
set(df.ev_type.unique())

{'Battery Electric Vehicle (BEV)', 'Plug-in Hybrid Electric Vehicle (PHEV)'}

In [17]:
set(df.cafv_type.unique())

{'Clean Alternative Fuel Vehicle Eligible',
 'Eligibility unknown as battery range has not been researched',
 'Not eligible due to low battery range'}